## /data/.cryptomator

In [ ]:
%%bash
FILE=/data/.cryptomator/auto-push.sh
mkdir -p $(dirname $FILE)

cat > $FILE << 'EOF'
#!/bin/bash
DIRS=(
  /data/.cryptomator
  /data/.manjaro
  /data/projects_ING/SMC
)
for DIR in "${DIRS[@]}"; do
  if [[ -d "$DIR/.git" ]]; then
    echo "=== 正在处理: $DIR ==="
    cd "$DIR" && git add -A && git diff --cached --quiet || git commit -m "$(date '+%Y-%m-%d')" && git push
  else
    echo "=== 跳过: $DIR (不是 git 仓库或未挂载) ==="
  fi
done
EOF
chmod +x $FILE


In [ ]:
FILE=/data/.cryptomator/auto-push.sh
ServiceFile=~/.config/systemd/user/git-backup.service
mkdir -p $(dirname $ServiceFile)
cat > $ServiceFile << EOF
[Unit]
Description=Git Auto Push
After=network.target

[Service]
ExecStart=$FILE
EOF


In [ ]:
TimerFile=~/.config/systemd/user/git-backup.timer
cat > $TimerFile << 'EOF'
[Unit]
Description=Git Auto Push Daily

[Timer]
OnCalendar=*-*-* 00:00:00
Persistent=true

[Install]
WantedBy=timers.target
EOF


In [ ]:
ServiceFile=~/.config/systemd/user/git-backup.service
TimerFile=~/.config/systemd/user/git-backup.timer
systemctl --user daemon-reload
systemctl --user enable --now $(basename $TimerFile)
systemctl --user status $(basename $TimerFile)
systemctl --user start $(basename $ServiceFile)
journalctl --user -u $(basename $ServiceFile) -n 50 --no-pager


In [1]:
%%bash
systemctl --user list-timers --all


NEXT                            LEFT LAST                         PASSED UNIT             ACTIVATES
Sun 2026-04-12 00:00:00 UTC 4h 23min Sat 2026-04-11 05:31:26 UTC 14h ago git-backup.timer git-backup.service

1 timers listed.
